In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import requests
import json
import os
import uuid
from datetime import datetime, timedelta
import random

# --- Configuration ---
OLLAMA_API_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3"  # Ensure this model is installed in Ollama
CSV_FILE_PATH = "live_shipment_data.csv"

# --- Data Generation Logic (from previous script) ---
def generate_and_save_data(num_records: int, file_path: str):
    """Generates a clean, interconnected dataset and saves it to a CSV file."""
    print(f"Generating {num_records} new shipment records...")
    bays = [{'ID': str(uuid.uuid4()), 'Code': f'BAY-0{i+1}'} for i in range(3)]
    products = [
        {'ID': str(uuid.uuid4()), 'Code': 'PROD-A95', 'Name': 'Unleaded Gasoline 95'},
        {'ID': str(uuid.uuid4()), 'Code': 'PROD-D50', 'Name': 'Diesel 50ppm'},
        {'ID': str(uuid.uuid4()), 'Code': 'PROD-JET', 'Name': 'Jet Fuel A1'}
    ]
    records = []
    for _ in range(num_records):
        bay = random.choice(bays)
        product = random.choice(products)
        entry_time = datetime.now() - timedelta(days=random.uniform(1, 60), hours=random.uniform(1, 23))
        exit_time = entry_time + timedelta(minutes=random.uniform(20, 60))
        gross_quantity = random.uniform(5000, 20000)
        duration_minutes = (exit_time - entry_time).total_seconds() / 60
        
        records.append({
            'ShipmentCode': f'SH-{random.randint(10000, 99999)}',
            'BayCode': bay['Code'],
            'BaseProductCode': product['Code'],
            'ProductName': product['Name'],
            'GrossQuantity': round(gross_quantity, 2),
            'FlowRate': round(gross_quantity / duration_minutes, 2) if duration_minutes > 0 else 0,
            'CreatedTime': entry_time,
            'ExitTime': exit_time
        })
    
    final_df = pd.DataFrame(records)
    final_df.to_csv(file_path, index=False)
    print(f"Successfully generated and saved data to '{file_path}'.")

# --- Ollama LLM Integration ---
def get_chart_spec_from_ollama(user_query: str, df: pd.DataFrame) -> dict:
    """Sends the user query to Ollama to get a JSON spec for a Matplotlib chart."""
    schema = df.dtypes.to_string()
    prompt = f"""
    You are an expert data analyst who generates specifications for Python's Matplotlib library.
    Your task is to convert a user's natural language query into a single, valid JSON object that describes a chart.

    The available data has the following columns:
    --- DATA SCHEMA ---
    {schema}
    --- END SCHEMA ---

    USER QUERY: "{user_query}"

    Based on the query and schema, generate a JSON object with the following keys:
    - "chart_type": "bar", "scatter", "line", or "histogram".
    - "x_column": The column for the x-axis. For a histogram, this is the column to analyze.
    - "y_column": The column for the y-axis. (Use null for histograms).
    - "aggregation": The pandas aggregation method to use if needed ("mean", "sum", "count"). Use "none" for scatter or line plots. For "bar" charts, you must provide a valid aggregation. If the user says "X by Y", assume they want the sum of Y for each X.
    - "title": A descriptive title for the chart.
    - "x_label": A label for the x-axis.
    - "y_label": A label for the y-axis.

    Example 1:
    User Query: "average flow rate by product"
    JSON Output: {{"chart_type": "bar", "x_column": "ProductName", "y_column": "FlowRate", "aggregation": "mean", "title": "Average Flow Rate by Product", "x_label": "Product", "y_label": "Average Flow Rate"}}

    Example 2:
    User Query: "scatter plot of quantity vs flowrate"
    JSON Output: {{"chart_type": "scatter", "x_column": "GrossQuantity", "y_column": "FlowRate", "aggregation": "none", "title": "Gross Quantity vs. Flow Rate", "x_label": "Gross Quantity (L)", "y_label": "Flow Rate (L/min)"}}

    Now, generate the JSON for the user's query. Return ONLY the single-line JSON object.
    """
    try:
        print("\n🤖 Asking the AI to design the chart...")
        payload = {"model": OLLAMA_MODEL, "prompt": prompt, "stream": False, "format": "json"}
        response = requests.post(OLLAMA_API_URL, json=payload, timeout=60)
        response.raise_for_status()
        
        response_text = json.loads(response.text).get("response", "{}")
        return json.loads(response_text)
        
    except requests.exceptions.RequestException as e:
        print(f"\n❌ Error connecting to Ollama: {e}")
        return None
    except (json.JSONDecodeError, AttributeError):
        print("\n❌ Error: The AI returned a malformed response. Please try a different query.")
        return None

# --- Matplotlib Chart Generation (CORRECTED LOGIC) ---
def generate_plot(df: pd.DataFrame, spec: dict):
    """Uses Matplotlib to generate and display a chart based on the AI's spec."""
    try:
        print("📊 Generating plot...")
        plt.style.use('seaborn-v0_8-whitegrid')
        fig, ax = plt.subplots(figsize=(10, 6))

        chart_type = spec.get("chart_type")
        x_col = spec.get("x_column")
        y_col = spec.get("y_column")

        # --- REFACTORED LOGIC ---
        # Prioritize chart_type and handle aggregations within each type.
        if chart_type == "bar":
            agg = spec.get("aggregation")
            # If the AI requests a bar chart but forgets the aggregation, default to 'sum'.
            if agg not in ["mean", "sum", "count"]:
                print(f"⚠️ AI did not specify an aggregation for a bar chart. Defaulting to 'sum'.")
                agg = "sum"
            
            if agg == 'count':
                plot_data = df.groupby(x_col)[x_col].count()
            else:
                plot_data = df.groupby(x_col)[y_col].agg(agg)
            
            plot_data.plot(kind='bar', ax=ax, rot=45)
            ax.set_xticklabels(ax.get_xticklabels(), ha="right")

        elif chart_type == "histogram":
            ax.hist(df[x_col], bins=20, edgecolor='black')

        elif chart_type == "scatter":
            ax.scatter(df[x_col], df[y_col], alpha=0.6)

        elif chart_type == "line":
            df_sorted = df.sort_values(by=x_col)
            ax.plot(df_sorted[x_col], df_sorted[y_col])
            
        else:
            print(f"❌ Unsupported or unknown chart type from AI: '{chart_type}'")
            return

        ax.set_title(spec.get("title", "Generated Chart"), fontsize=16, fontweight='bold')
        ax.set_xlabel(spec.get("x_label", x_col), fontsize=12)
        ax.set_ylabel(spec.get("y_label", y_col), fontsize=12)
        
        plt.tight_layout()
        plt.show()

    except KeyError as e:
        print(f"\n❌ Charting Error: The AI specified a column that doesn't exist: {e}")
    except Exception as e:
        print(f"\n❌ An unexpected error occurred during chart generation: {e}")


# --- Main Application Loop ---
def main():
    """Main function to run the AI reporter application."""
    if not os.path.exists(CSV_FILE_PATH):
        print(f"Data file '{CSV_FILE_PATH}' not found.")
        if input("Would you like to generate a new sample data file? (y/n): ").lower() == 'y':
            generate_and_save_data(10000, CSV_FILE_PATH)
        else:
            print("Exiting. Please create the data file to proceed.")
            return
            
    try:
        df = pd.read_csv(CSV_FILE_PATH)
        df['CreatedTime'] = pd.to_datetime(df['CreatedTime'])
        df['ExitTime'] = pd.to_datetime(df['ExitTime'])
    except Exception as e:
        print(f"Error loading or parsing {CSV_FILE_PATH}: {e}")
        return

    print("\n--- AI Matplotlib Reporter ---")
    print("Type your request for a chart (e.g., 'total quantity by product'), or type 'exit' to quit.")

    while True:
        user_input = input("\n> What chart would you like to see? ")
        if user_input.lower() == 'exit':
            break
        
        chart_spec = get_chart_spec_from_ollama(user_input, df)
        
        if chart_spec:
            generate_plot(df, chart_spec)

    print("Application closed.")

if __name__ == "__main__":
    main()

